# Aula extra ? Redes neurais em pr?tica: regress?o e classifica??o

Neste laborat?rio vamos observar uma **MLP aprendendo uma rela??o cont?nua** e uma **CNN classificando imagens**. Ao final, um gloss?rio conecta as arquiteturas citadas no curso aos tipos de problema em que costumam ser ?teis.

> Os dados de seguro usados na primeira parte s?o **sint?ticos e did?ticos**. Idade sozinha n?o determina pre?o real. CIFAR-10 ? um conjunto de imagens de refer?ncia, pequeno para padr?es atuais, mas o treino ainda pode levar alguns minutos.


## Objetivos de aprendizagem

- Distinguir regress?o (sa?da num?rica) de classifica??o (r?tulo/categoria).
- Ver como a curva de uma MLP muda ao longo do treinamento.
- Reconhecer pixels como uma entrada com estrutura espacial e entender o papel de uma CNN.
- Relacionar MLP, CNN 1D, CNN 2D, LSTM e autoencoder ao problema adequado.

### Mapa do laborat?rio

1. Regress?o de custo de seguro por idade com uma MLP e snapshots em diferentes ?pocas.
2. Classifica??o de imagens CIFAR-10 com uma CNN pequena.
3. Gloss?rio e perguntas para discuss?o.


## 1. Regress?o: custo ilustrativo do seguro por idade

A tarefa ? estimar um valor cont?nuo (`custo`). Para visualizar aprendizado com clareza, simulamos custos com uma tend?ncia curva, ru?do e observa??es. A divis?o treino/teste ? fixa. Padronizamos a idade usando somente o conjunto de treino; isso ajuda a otimiza??o da rede.

**Aten??o:** a curva sint?tica serve para ensinar o mecanismo de ajuste. N?o representa uma tabela atuarial, n?o inclui fatores importantes e n?o deve ser usada para precificar seguros reais.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
idade = np.sort(rng.uniform(18, 70, size=180))
# Rela??o curva sint?tica, com ru?do: custo em unidades monet?rias ilustrativas.
custo = 450 + 18 * idade + 0.42 * idade**2 + rng.normal(0, 420, size=idade.size)
X = idade.reshape(-1, 1)
y = custo

ordem = rng.permutation(len(y))
n_treino = int(0.8 * len(y))
idx_treino, idx_teste = ordem[:n_treino], ordem[n_treino:]
media_idade = X[idx_treino].mean(axis=0)
dp_idade = X[idx_treino].std(axis=0)
X_treino = (X[idx_treino] - media_idade) / dp_idade
X_teste = (X[idx_teste] - media_idade) / dp_idade
print(f"Treino: {len(idx_treino)} observa??es | teste: {len(idx_teste)}")


### Inspe??o dos dados

Antes de treinar, olhe a nuvem de pontos. Uma MLP n?o recebe a f?rmula quadr?tica acima: ela ajusta seus pesos a partir de exemplos. Usamos uma camada oculta com ativa??o ReLU e sa?da linear, apropriada para prever um n?mero.


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(X[idx_treino, 0], y[idx_treino], alpha=.7, label="Treino")
plt.scatter(X[idx_teste, 0], y[idx_teste], alpha=.8, label="Teste")
plt.xlabel("Idade (anos)"); plt.ylabel("Custo sint?tico")
plt.title("Dados ilustrativos de custo do seguro por idade")
plt.legend(); plt.grid(alpha=.2); plt.show()


### Treinar passo a passo e salvar ?pocas

`partial_fit` executa uma ?poca por chamada. Isso permite guardar previs?es intermedi?rias e observar a curva evoluindo. Os gr?ficos usam uma grade ordenada de idades para a linha prevista; os pontos de teste permanecem fora do treinamento.


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

modelo_reg = MLPRegressor(
    hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
    learning_rate_init=0.01, alpha=0.001, max_iter=1, warm_start=True,
    random_state=SEED
)
epocas = [1, 5, 20, 100, 400]
curva = np.linspace(18, 70, 250).reshape(-1, 1)
curva_padronizada = (curva - media_idade) / dp_idade
predicoes_epoca = {}
perda = []

for epoca in range(1, max(epocas) + 1):
    modelo_reg.partial_fit(X_treino, y[idx_treino])
    perda.append(modelo_reg.loss_)
    if epoca in epocas:
        predicoes_epoca[epoca] = modelo_reg.predict(curva_padronizada)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
for ax, epoca in zip(axes.flat, epocas):
    ax.scatter(X[idx_treino, 0], y[idx_treino], s=18, alpha=.45, label="Treino")
    ax.scatter(X[idx_teste, 0], y[idx_teste], s=22, alpha=.8, label="Teste")
    ax.plot(curva[:, 0], predicoes_epoca[epoca], color="crimson", lw=2, label="MLP")
    ax.set_title(f"?poca {epoca}"); ax.grid(alpha=.2)
    ax.set_xlabel("Idade (anos)"); ax.set_ylabel("Custo sint?tico")
axes.flat[-1].axis("off")
axes.flat[0].legend()
fig.suptitle("A curva prevista pela MLP durante o aprendizado", fontsize=15)
plt.tight_layout(); plt.show()


**Leitura do gr?fico:** nas primeiras ?pocas, os pesos ainda n?o capturam bem o padr?o. Com mais atualiza??es, a curva tende a acompanhar melhor a tend?ncia dos dados. Treinar por mais tempo n?o garante melhora: pode haver oscila??o ou sobreajuste.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(perda) + 1), perda)
plt.xlabel("?poca"); plt.ylabel("Perda de treinamento (MSE interno)")
plt.title("Perda da MLP durante o treinamento"); plt.grid(alpha=.25); plt.show()

pred_teste_reg = modelo_reg.predict(X_teste)
rmse = mean_squared_error(y[idx_teste], pred_teste_reg) ** 0.5
print(f"MAE:  {mean_absolute_error(y[idx_teste], pred_teste_reg):,.1f}")
print(f"RMSE: {rmse:,.1f}")
print(f"R?:   {r2_score(y[idx_teste], pred_teste_reg):.3f}")


### Perguntas r?pidas

1. O que muda entre a previs?o da ?poca 1 e da ?poca 100?
2. Por que a avalia??o usa observa??es que n?o participaram do ajuste?
3. Se o R? for baixo ou negativo no teste, o que isso sugere?
4. Que vari?veis reais, al?m da idade, seriam necess?rias para estudar pre?o de seguro com responsabilidade?


## 2. Classifica??o de imagens: CIFAR-10 com CNN

CIFAR-10 cont?m imagens coloridas pequenas (32?32 pixels) divididas em dez classes. Uma CNN 2D usa filtros que percorrem regi?es vizinhas da imagem para aprender padr?es locais, como bordas e texturas. A cabe?a final produz dez escores, um por classe.

A primeira execu??o baixa o conjunto via Keras/TensorFlow. Para reduzir o tempo em computadores modestos, o exemplo treina inicialmente com uma amostra configur?vel. Aumente a amostra e as ?pocas para uma experi?ncia mais completa.


In [ ]:
# Se necess?rio, instale em uma c?lula separada: %pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
nomes_classes = ["avi?o", "autom?vel", "p?ssaro", "gato", "cervo",
                 "cachorro", "sapo", "cavalo", "navio", "caminh?o"]
(x_treino_img, y_treino_img), (x_teste_img, y_teste_img) = keras.datasets.cifar10.load_data()
y_treino_img = y_treino_img.ravel(); y_teste_img = y_teste_img.ravel()
print("Treino original:", x_treino_img.shape, "Teste:", x_teste_img.shape)


In [ ]:
# Ajuste estes limites para equilibrar tempo de execu??o e qualidade did?tica.
N_TREINO = 12000
N_TESTE = 3000
EPOCAS_CNN = 8

xtr = x_treino_img[:N_TREINO].astype("float32") / 255.0
ytr = y_treino_img[:N_TREINO]
xte = x_teste_img[:N_TESTE].astype("float32") / 255.0
yte = y_teste_img[:N_TESTE]

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_treino_img[i])
    ax.set_title(nomes_classes[y_treino_img[i]])
    ax.axis("off")
plt.suptitle("Exemplos do CIFAR-10"); plt.tight_layout(); plt.show()


### Arquitetura e treinamento

A normaliza??o transforma intensidades de pixels de 0?255 para 0?1. As convolu??es e o pooling extraem e resumem padr?es espaciais; `Flatten` converte os mapas em vetor para a camada de classifica??o. A sa?da usa logits e a fun??o de perda aplica `from_logits=True`.


In [ ]:
modelo_cnn = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10)  # logits: um escore por classe
])
modelo_cnn.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
historico = modelo_cnn.fit(
    xtr, ytr, validation_split=0.15, epochs=EPOCAS_CNN,
    batch_size=64, verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(historico.history["accuracy"], label="Treino")
axes[0].plot(historico.history["val_accuracy"], label="Valida??o")
axes[0].set(title="Acur?cia", xlabel="?poca", ylabel="Acur?cia"); axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].plot(historico.history["loss"], label="Treino")
axes[1].plot(historico.history["val_loss"], label="Valida??o")
axes[1].set(title="Perda", xlabel="?poca", ylabel="Cross-entropy"); axes[1].legend(); axes[1].grid(alpha=.2)
plt.tight_layout(); plt.show()

perda_teste, acuracia_teste = modelo_cnn.evaluate(xte, yte, verbose=0)
print(f"Acur?cia no subconjunto de teste: {acuracia_teste:.3f}")


### Inspecionar previs?es e matriz de confus?o


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

prob = tf.nn.softmax(modelo_cnn.predict(xte[:500], verbose=0), axis=1).numpy()
pred = prob.argmax(axis=1)
print(classification_report(yte[:500], pred, target_names=nomes_classes, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    yte[:500], pred, display_labels=nomes_classes, xticks_rotation=45,
    cmap="Blues", normalize="true"
)
plt.title("Matriz de confus?o normalizada (amostra do teste)"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(xte[i])
    ax.set_title(f"Real: {nomes_classes[yte[i]]}\nPrev.: {nomes_classes[pred[i]]}",
                 color="green" if pred[i] == yte[i] else "crimson", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()


## Conceitos de Deep Learning para interpretar os exemplos

Os conceitos abaixo seguem a apresenta??o de Goodfellow, Bengio e Courville. O livro ? uma refer?ncia acad?mica confi?vel para fundamentos e arquiteturas cl?ssicas; suas recomenda??es t?cnicas s?o de 2016, ent?o detalhes de bibliotecas e pr?ticas atuais devem ser conferidos tamb?m na documenta??o oficial contempor?nea.

### Representa??o aprendida e camadas

Uma rede feedforward aplica transforma??es sucessivas ? entrada. Cada camada oculta aprende uma representa??o intermedi?ria; as ativa??es n?o lineares permitem modelar rela??es que uma ?nica transforma??o linear n?o representa. Na MLP de regress?o, a sa?da linear produz um valor cont?nuo; na CNN de imagens, convolu??es formam mapas de caracter?sticas antes da camada de classifica??o. (Cap. 6: [Deep Feedforward Networks](https://www.deeplearningbook.org/contents/mlp.html))

### Fun??o de perda, gradiente e ?pocas

O treinamento ajusta par?metros para reduzir uma fun??o de perda nos dados de treino. O gradiente indica como alterar os par?metros para reduzir essa perda; uma ?poca corresponde a uma passagem de treinamento pelos exemplos. A curva de treino sozinha n?o mede generaliza??o: acompanhe valida??o e reserve o teste para avalia??o final. (Cap. 8: [Optimization for Training Deep Models](https://www.deeplearningbook.org/contents/optimization.html); Cap. 11: [Practical Methodology](https://www.deeplearningbook.org/contents/guidelines.html))

### Generaliza??o e regulariza??o

O objetivo ? que o modelo funcione em exemplos ainda n?o vistos. Se o erro de treino continua caindo enquanto o erro de valida??o piora, h? sinal de sobreajuste. Regulariza??o, arquitetura adequada, parada antecipada e dados representativos podem ajudar; nenhuma t?cnica substitui uma parti??o de avalia??o bem planejada. (Cap. 5: [Machine Learning Basics](https://www.deeplearningbook.org/contents/ml.html); Cap. 7: [Regularization for Deep Learning](https://www.deeplearningbook.org/contents/regularization.html))

### Vi?s indutivo: por que arquiteturas diferentes?

Uma arquitetura incorpora pressupostos sobre os dados. MLP trata a entrada como vetor e n?o explora explicitamente vizinhan?a espacial; CNN usa filtros locais e compartilhados, uma escolha natural para imagens e sinais; RNN/LSTM processa sequ?ncias mantendo estado ao longo dos passos. Esse vi?s pode ajudar quando combina com a estrutura do problema, mas n?o garante melhor desempenho. (Cap. 9: [Convolutional Networks](https://www.deeplearningbook.org/contents/convnets.html); Cap. 10: [Sequence Modeling](https://www.deeplearningbook.org/contents/rnn.html))

### Reconstru??o e autoencoders

Um autoencoder aprende um codificador, uma representa??o comprimida e um decodificador que tenta reconstruir a entrada. O gargalo for?a uma representa??o compacta. Usar erro de reconstru??o para sinalizar anomalias ? uma estrat?gia poss?vel, mas limiar, distribui??o de treino e tipos de anomalia precisam ser validados; erros altos n?o provam falha. (Cap. 14: [Autoencoders](https://www.deeplearningbook.org/contents/autoencoders.html))

## 3. Gloss?rio em blocos: arquitetura, problema e dataset

Cada bloco liga uma arquitetura a um problema concreto e a um conjunto de dados que pode ser carregado para experimentar. Os datasets s?o exemplos de estudo; verifique licen?a, vers?o e forma de divis?o antes de usar em um projeto.

### MLP ? dados tabulares

**Use quando:** cada exemplo pode ser representado por uma linha de atributos num?ricos/categ?ricos e a sa?da ? uma categoria ou um valor cont?nuo.

**Por que:** camadas densas combinam os atributos e aprendem rela??es n?o lineares. Para dados tabulares, compare sempre com modelos como regress?o linear, Random Forest ou Gradient Boosting; MLP n?o ? automaticamente superior.

**Dataset para praticar:** [Diabetes (regress?o), do scikit-learn](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) tem atributos cl?nicos num?ricos e alvo cont?nuo. Tamb?m ? poss?vel usar o [Wine Quality, do UCI](https://archive.ics.uci.edu/dataset/186/wine+quality), para regress?o ou classifica??o ordinal.

**Exemplo:** prever progress?o do diabetes usando atributos cl?nicos.

```python
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)
mlp = make_pipeline(
    StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16),
                                   max_iter=1000, random_state=42)
)
mlp.fit(X_tr, y_tr)
print("R? teste:", mlp.score(X_te, y_te))
```

### CNN 1D ? sinais e s?ries temporais

**Use quando:** a entrada ? uma sequ?ncia ordenada (por exemplo, leituras consecutivas de sensores) e padr?es locais ao longo do tempo ajudam a prever ou classificar.

**Por que:** filtros aprendem formas locais, como picos, quedas e oscila??es, percorrendo a sequ?ncia. CNN 1D n?o substitui o cuidado com ordem temporal: separe treino e teste por tempo ou equipamento para evitar vazamento.

**Dataset para praticar:** [Jena Climate](https://www.tensorflow.org/tutorials/structured_data/time_series) tem medi??es meteorol?gicas a cada 10 minutos e ? usado no tutorial oficial de s?ries temporais do TensorFlow. Para contexto industrial, o [MetroPT-3 no Kaggle](https://www.kaggle.com/datasets/joebeachcapital/metropt-3-dataset) cont?m telemetria de compressor; confira eventos e timestamps antes de criar r?tulos.

**Exemplo:** criar janelas de temperaturas passadas para prever a temperatura no pr?ximo intervalo.

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

cnn1d = keras.Sequential([
    layers.Input(shape=(24, 5)),  # 24 instantes, 5 sinais
    layers.Conv1D(32, kernel_size=3, activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(1)  # sa?da num?rica: regress?o
])
cnn1d.compile(optimizer="adam", loss="mse")
# X_janelas: (amostras, 24, 5); y_proximo: pr?ximo valor-alvo
# cnn1d.fit(X_janelas, y_proximo, ...)
```

### CNN 2D ? imagens

**Use quando:** cada observa??o ? uma imagem e a posi??o relativa dos pixels carrega informa??o.

**Por que:** convolu??es exploram vizinhan?as de pixels e detectam padr?es visuais em diferentes regi?es da imagem.

**Dataset para praticar:** [CIFAR-10](https://www.tensorflow.org/datasets/catalog/cifar10) re?ne 60 mil imagens coloridas 32?32 em dez classes; ? o dataset usado na se??o anterior. Para um primeiro exerc?cio menor, [MNIST](https://www.tensorflow.org/datasets/catalog/mnist) tem imagens de d?gitos escritos ? m?o.

**Exemplo:** classificar a imagem como avi?o, carro, p?ssaro, gato e outras classes do CIFAR-10.

```python
cnn2d = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation="softmax")
])
# cnn2d.compile(...); cnn2d.fit(imagens_treino, classes_treino, ...)
```

### LSTM ? sequ?ncias com depend?ncia temporal

**Use quando:** a ordem dos passos importa e a previs?o pode depender do contexto acumulado ao longo de v?rios instantes.

**Por que:** a camada recorrente atualiza um estado interno a cada passo, carregando informa??o relevante dos passos anteriores. Para sequ?ncias curtas, compare tamb?m com CNN 1D e baselines simples.

**Dataset para praticar:** o [Jena Climate do tutorial TensorFlow](https://www.tensorflow.org/tutorials/structured_data/time_series) permite prever temperatura, umidade ou press?o a partir do hist?rico.

**Exemplo:** prever temperatura uma hora ? frente usando 24 observa??es hor?rias de v?rias medi??es meteorol?gicas.

```python
lstm = keras.Sequential([
    layers.Input(shape=(24, 5)),
    layers.LSTM(32),
    layers.Dense(1)
])
lstm.compile(optimizer="adam", loss="mse")
# lstm.fit(janelas_treino, temperatura_futura, ...)
```

### Autoencoder ? representa??o e anomalias

**Use quando:** h? muitos exemplos do comportamento normal e poucos r?tulos confi?veis de falha; tamb?m pode servir para aprender representa??es compactas.

**Por que:** aprende a comprimir e reconstruir entradas. Se treinado principalmente com dados normais, uma reconstru??o ruim pode sinalizar um caso at?pico que merece investiga??o. O erro de reconstru??o ? um alerta, n?o confirma??o de defeito.

**Dataset para praticar:** [AI4I 2020 Predictive Maintenance (UCI)](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) ? sint?tico e traz leituras de processo e r?tulos de falha para comparar scores; para o exerc?cio n?o supervisionado, treine o autoencoder apenas nas linhas normais.

**Exemplo:** reconstruir cinco sensores de um ciclo de m?quina e inspecionar ciclos cujo erro m?dio seja alto.

```python
autoencoder = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(3, activation="relu"),
    layers.Dense(2, activation="relu"),  # gargalo
    layers.Dense(3, activation="relu"),
    layers.Dense(5)  # reconstru??o
])
autoencoder.compile(optimizer="adam", loss="mse")
# ajuste apenas em X_normal; depois calcule erro por amostra:
# erro = ((X_avaliar - autoencoder.predict(X_avaliar)) ** 2).mean(axis=1)
```

### Resumo da escolha

- **Linha de atributos tabulares ? MLP** como candidato; compare com m?todos cl?ssicos.
- **Imagem ? CNN 2D** para explorar padr?es espaciais.
- **Sinal sequencial ? CNN 1D** para formas locais; **LSTM** quando contexto temporal acumulado for relevante.
- **Poucos r?tulos de falha e exemplos normais dispon?veis ? autoencoder** para priorizar inspe??o.

A arquitetura ? uma hip?tese inicial. A decis?o final depende de valida??o apropriada, m?tricas, custo de erro, volume de dados, lat?ncia e explicabilidade.


## Refer?ncias e cr?ditos

- Goodfellow, Ian; Bengio, Yoshua; Courville, Aaron. *Deep Learning*. MIT Press, 2016. Livro completo dispon?vel no [site oficial](https://www.deeplearningbook.org/). Cap?tulos usados: [6 ? redes feedforward](https://www.deeplearningbook.org/contents/mlp.html), [7 ? regulariza??o](https://www.deeplearningbook.org/contents/regularization.html), [8 ? otimiza??o](https://www.deeplearningbook.org/contents/optimization.html), [9 ? redes convolucionais](https://www.deeplearningbook.org/contents/convnets.html), [10 ? modelagem de sequ?ncias](https://www.deeplearningbook.org/contents/rnn.html), [11 ? metodologia pr?tica](https://www.deeplearningbook.org/contents/guidelines.html), [14 ? autoencoders](https://www.deeplearningbook.org/contents/autoencoders.html). A p?gina oficial informa autores, MIT Press, ano e acesso online gratuito.
- CIFAR-10 ? Alex Krizhevsky, *Learning Multiple Layers of Features from Tiny Images* (2009); dispon?vel em `keras.datasets.cifar10`.
- Scikit-learn ? [`MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).
- TensorFlow/Keras ? [Convolutional layers](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) e [CIFAR-10 tutorial](https://www.tensorflow.org/tutorials/images/cnn).
- Os dados da regress?o de seguro foram gerados neste notebook para fins did?ticos; n?o s?o dados reais de seguradora.
